# Unmet healthcare needs in South Korea, 2020-2023: a four-wave latent transition analysis

Reproducible companion to the manuscript submitted to *Health Policy*.

This notebook runs the complete analysis: it builds the person-wave panel, selects the number of latent statuses, fits the latent transition model with bootstrap confidence intervals, estimates the covariate models, and writes every table, figure and quoted number.

**Two ways to run it**

| Situation | What to do | What you get |
|---|---|---|
| You have no Korea Health Panel licence | run the cells in order and leave `USE_REAL_KHP = False` | the whole pipeline on a synthetic replica of the panel; estimates close to, but not identical with, the published ones |
| You have an approved copy of the KHP files | set `USE_REAL_KHP = True` and give `KHP_RAW_DIR` | every number in the manuscript |

The Korea Health Panel is released to named applicants under a data-use agreement that forbids redistribution, so the microdata are not in the repository and cannot be downloaded from this notebook. Apply at <https://www.khp.re.kr>.

Runtime: about 10 minutes with the default settings, or 50-70 minutes with the full 400 bootstrap replications.

## 1. Install the dependencies

In [ ]:
!pip -q install pyreadstat==1.3.6 2>&1 | tail -2
import numpy, pandas, scipy, matplotlib, pyreadstat, sys
print('python     ', sys.version.split()[0])
for m in (numpy, pandas, scipy, matplotlib, pyreadstat):
    print(f'{m.__name__:<12}', m.__version__)

## 2. Get the code

Replace `REPO_URL` with the repository you pushed this package to. If you would rather upload the folder by hand, skip this cell and set `PROJECT` to the uploaded path.

In [ ]:
import os, pathlib, subprocess

REPO_URL = 'https://github.com/<your-account>/HealthPolicy_UnmetNeed_LTA.git'
PROJECT = pathlib.Path('/content/HealthPolicy_UnmetNeed_LTA')

if not PROJECT.exists():
    try:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT)],
                       check=True)
    except Exception as exc:
        print('clone failed:', exc)
        print('Upload the project folder to /content instead, then set PROJECT to it.')

print(PROJECT, 'exists:', PROJECT.exists())
if PROJECT.exists():
    print(sorted(p.name for p in PROJECT.iterdir()))

## 3. Choose the data source

`USE_REAL_KHP = True` needs the four annual SPSS files in a folder laid out as
`KHP+2020_spss/b_ind.sav`, `KHP+2021_spss/c_ind.sav`, `KHP+2022_spss/d_ind.sav`,
`KHP+2023_spss/e_ind.sav`. Mounting your own Drive is the usual way to reach them.

In [ ]:
USE_REAL_KHP = False          # True only if you hold an approved copy of the KHP files
QUICK = True                  # 40 bootstrap replications instead of 400

if USE_REAL_KHP:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['KHP_RAW_DIR'] = '/content/drive/MyDrive/medical_pannel'   # edit this

os.chdir(PROJECT / 'src')
print('working directory:', os.getcwd())
!python 00_check_environment.py

## 4. Run the pipeline

`run_all.py` executes the steps in order and writes `output/logs/run_log.txt`.
Every stochastic step is seeded, so a second run on the same data gives the same
estimates.

In [ ]:
flags = []
if not USE_REAL_KHP:
    flags.append('--synthetic')
if QUICK:
    flags.append('--quick')

!python run_all.py {' '.join(flags)}

## 5. Inspect the model step by step

The cells below reproduce the three results the manuscript rests on, using the
modules directly rather than the scripts, so you can change the specification and
see what happens.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT / 'src'))
import numpy as np, pandas as pd
from khp_lta import config as cfg, lta
from khp_lta.build import indicator_array

panel = pd.read_csv(cfg.PANEL_CSV)
Y, w, ids = indicator_array(panel)
print(f'{Y.shape[0]:,} adults x {Y.shape[1]} waves x {Y.shape[2]} indicators')
print(f'person-waves with the module answered: {int((~np.isnan(Y)).any(axis=2).sum()):,}')

obs = pd.DataFrame(
    100 * np.nanmean(Y, axis=0), index=cfg.YEARS, columns=cfg.INDICATORS)
obs.round(2)

### 5a. How many latent statuses?

Fewer random starts than the published run, so this finishes in a couple of minutes.

In [ ]:
rows = []
for K in (1, 2, 3, 4, 5):
    fit = lta.fit_lta(Y, w, K, n_starts=8, seed=cfg.RANDOM_SEED + K,
                      screen_iter=200, n_polish=2, max_iter=6000, tol=1e-5)
    rows.append({'K': K, 'log_likelihood': round(fit.loglik, 1),
                 'parameters': fit.n_params, 'BIC': round(fit.bic, 1),
                 'entropy_R2': round(fit.entropy_r2(), 3) if K > 1 else None,
                 'smallest_status_%': round(100 * fit.prevalence.mean(axis=0).min(), 2)})
pd.DataFrame(rows)

### 5b. What the four statuses are

Each column is the probability of reporting that barrier given membership of the
status. The statuses are ordered by overall burden, lowest first.

In [ ]:
fit = lta.fit_lta(Y, w, 4, n_starts=20, seed=cfg.RANDOM_SEED,
                  screen_iter=300, n_polish=3, max_iter=20000, tol=1e-7)
names = ['No unmet need', 'Cost-driven', 'Non-financial', 'Compound']
print(f'log-likelihood {fit.loglik:,.2f}   entropy R2 {fit.entropy_r2():.3f}')

display(pd.DataFrame(fit.rho, index=[cfg.INDICATOR_LABELS[i] for i in cfg.INDICATORS],
                     columns=names).round(3))
display(pd.DataFrame(100 * fit.prevalence, index=cfg.YEARS, columns=names).round(1))

### 5c. Where people go the following year

In [ ]:
for t in range(cfg.N_WAVES - 1):
    print(f'{cfg.YEARS[t]} to {cfg.YEARS[t + 1]}')
    display(pd.DataFrame(fit.tau[t], index=names, columns=names).round(3))

## 6. Look at the results

Everything the manuscript reports is now in `output/`.

In [ ]:
import json
from IPython.display import Image, display

for name in sorted(p.name for p in cfg.TABLES_DIR.glob('*.csv')):
    print(name)

print()
display(pd.read_csv(cfg.TABLES_DIR / 'table2_measurement_model.csv'))
display(pd.read_csv(cfg.TABLES_DIR / 'table3_transition_matrices.csv'))

png = cfg.FIGURES_DIR / 'figure1_status_prevalence_and_persistence.png'
if png.exists():
    display(Image(filename=str(png)))

if cfg.NUMBERS_JSON.exists():
    print(json.dumps(json.loads(cfg.NUMBERS_JSON.read_text(encoding='utf-8')),
                     indent=2)[:3000])

## 7. Save the outputs back to Drive (optional)

In [ ]:
SAVE_TO_DRIVE = False
DEST = '/content/drive/MyDrive/HealthPolicy_UnmetNeed_LTA_output'

if SAVE_TO_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.copytree(cfg.OUTPUT_DIR, DEST, dirs_exist_ok=True)
    print('copied output/ to', DEST)